### CRISP-DM Phase 5.2 - Evaluation : Statistical significance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load the datasets
correlation_country = pd.read_csv('outputs/correlation_country.csv')
correlation_continent = pd.read_csv('outputs/correlation_continent.csv')

Statistical significance

In [ ]:
def statistical_significance(row):
    if row['p_value'] >= 0.05:
        return 'Not significant'
    elif row['Rho'] > 0.3:
        return 'Strong positive relationship'
    elif row['Rho'] < -0.3:
        return 'Strong negative relationship'
    else:
        return 'Weak relationship'

correlation_country['Result'] = correlation_country.apply(statistical_significance, axis=1)
print(f"Country-level correlation results:\n{correlation_country['Result'].value_counts()}")

correlation_continent['Result'] = correlation_continent.apply(statistical_significance, axis=1)
print(f"Continent-level correlation results:\n{correlation_continent['Result'].value_counts()}")

In [ ]:
# By hazard
print(f"Country-level result classification by hazard:\n{correlation_country.groupby(['Hazard', 'Result'])['Country'].count().unstack(fill_value=0)}")
print(f"Continental-level result classification by hazard:\n{correlation_continent.groupby(['Hazard', 'Result'])['Continent'].count().unstack(fill_value=0)}")

Correlation analysis

In [ ]:
## Distribution of rho
# Country
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correlation_country['Rho'], bins=30, edgecolor='white')
axes[0].set_xlabel('Rho')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of country-level correlation coefficients')

significant = correlation_country[correlation_country['p_value'] < 0.05]['Rho']
non_significant = correlation_country[correlation_country['p_value'] >= 0.05]['Rho']
axes[1].hist(non_significant, bins=20, alpha=0.6, color='grey', edgecolor='white', label='Not significant')
axes[1].hist(significant, bins=20, alpha=0.6, edgecolor='white', label='Significant')
axes[1].set_xlabel('Rho')
axes[1].set_ylabel('Count')
axes[1].set_title('Significant vs non-significant country-level correlations')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/5_rho_distribution_country.png', dpi=150, bbox_inches='tight')
plt.close()

# Continent
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correlation_continent['Rho'], bins=15, edgecolor='white')
axes[0].set_xlabel('Rho')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of continent-level correlation coefficients')

significant = correlation_continent[correlation_continent['p_value'] < 0.05]['Rho']
non_significant = correlation_continent[correlation_continent['p_value'] >= 0.05]['Rho']
axes[1].hist(non_significant, bins=10, alpha=0.6, color='grey', edgecolor='white', label='Not significant')
axes[1].hist(significant, bins=10, alpha=0.6, edgecolor='white', label='Significant')
axes[1].set_xlabel('Rho')
axes[1].set_ylabel('Count')
axes[1].set_title('Significant vs non-significant continental-level correlations')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/5_rho_distribution_continent.png', dpi=150, bbox_inches='tight')
plt.close()

In [ ]:
## Mean rho per hazard 
fig, axes = plt.subplots(2, 1, figsize=(10, 5))

# Country
hazard_mean_rho_country = correlation_country.groupby('Hazard')['Rho'].mean().sort_values()
colors_country = ['red' if r < 0 else 'green' for r in hazard_mean_rho_country.values]
axes[0].barh(hazard_mean_rho_country.index, hazard_mean_rho_country.values, color=colors_country, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Average country-level correlation by hazard category')
axes[0].set_xlabel('Mean rho')

# Continent
hazard_mean_rho_continent = correlation_continent.groupby('Hazard')['Rho'].mean().sort_values()
colors_continent = ['red' if r < 0 else 'green' for r in hazard_mean_rho_continent.values]
axes[1].barh(hazard_mean_rho_continent.index, hazard_mean_rho_continent.values, color=colors_continent, edgecolor="white")
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Average continental-level correlation by hazard category')
axes[1].set_xlabel('Mean rho')

plt.tight_layout()
plt.subplots_adjust(hspace=0.5)
plt.savefig("outputs/5_mean_rho.png", dpi=150, bbox_inches="tight")
plt.close()

In [ ]:
# Correlation comparison
country_summary = correlation_country.groupby('Hazard')['Rho'].mean().rename('country_mean_rho')
continent_summary = correlation_continent.groupby('Hazard')['Rho'].mean().rename('continent_mean_rho')
comparison = pd.concat([country_summary, continent_summary], axis=1)
comparison['difference'] = comparison['continent_mean_rho'] - comparison['country_mean_rho']
print(comparison.sort_values('difference', ascending=False))

In [ ]:
## Countries with highest and lowest correlations for each hazard
for hazard in correlation_country['Hazard'].unique():
    subset = correlation_country[(correlation_country['Hazard'] == hazard) & correlation_country['Rho'].notna()
                                 ].sort_values('Rho', ascending=False)    
    print(f"Strongest positive correlations for {hazard}:")
    print(subset.head(5)[['Country', 'Rho', 'p_value']])
    print(f"Strongest negative correlations for {hazard}:")
    print(subset.tail(5)[['Country', 'Rho', 'p_value']])

In [ ]:
# Most reccuring countries 
strong_positive = correlation_country[(correlation_country['Rho'] > 0.3) & (correlation_country['p_value'] < 0.05)]
print(strong_positive['Country'].value_counts().head(15))
strong_negative = correlation_country[(correlation_country['Rho'] < -0.3) & (correlation_country['p_value'] < 0.05)]
print(strong_negative['Country'].value_counts().head(15))